In [2]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

For the rest of this class I want to brain dump everything we talked about today in class so I have a good idea of what to do going forward.

I need to rethink how accuracy is done:
- Look at volatility for each ticker and determine accurate 1 or not accurate 0 based on the specific ticker rather than the global bounds I defined
- If the row is positive sentiment then for now if the return is positive in the right time frame accuracy is 1 - if need to use volatility for better bounding revisit later after first pass
- same logic if the row is negative sentiment
- if its one of the 20,000 neutral sentiment rows, then whether they are accurate or not depends if the stock price in the right time frame stays within some ticker specific bounds.

Modeling use unsupervised learning - kmeans clustering. Theres 2 aspects to this project clustering and accuracy they are 2 separate things.

Modeling - what would the user like to see:
- For kmeans focus on the sentiment and the ticker and leave the accuracy to be a separate thing (that means drop signal)
- It would be cool to have a general dashboard widget where you can see the clusters of publishers based on their sentiment articles published what else?
- The real deal is a widget showing clustering of tickers

The vision is basically as an investor I look up the ticker im interested in (maybe dropdown of allowed tickers), and I can see the ticker in the cluster and then as a separate thing I can see all the authors that have written about the ticker, how many articles, their accuracy and anything else?

First step is to calculate the volatility for each ticker and then go through and redo the accuracy column
Second step is build multiple clusters
Third step is to turn them into widgets and figure out how to make it interactive select ticker -> show authors sorted by accuracy -> select author and show their stats for that specific ticker. 

In [3]:
data = pd.read_csv('modeling.csv')
data.head()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m,article_text,finbert_label,finbert_score,sentiment_encoded,long_term,movement,accuracy
0,"UBS Maintains Buy on Adobe, Raises Price Targe...",https://www.benzinga.com/news/20/06/16202690/u...,Benzinga Newsdesk,2020-06-08,ADBE,0,2428,1,0.008950,0.150586,0.237493,Never miss a trade again with the fastest news...,neutral,0.932860,0,0,1,0
1,Shares of several technology companies are tra...,https://www.benzinga.com/wiim/20/05/16075931/s...,Benzinga Newsdesk,2020-05-20,ADBE,1,94,0,-0.022052,0.115684,0.197690,Never miss a trade again with the fastest news...,neutral,0.932860,0,0,1,0
2,"Benzinga's Top Upgrades, Downgrades For May 14...",https://www.benzinga.com/markets/penny-stocks/...,Lisa Levin,2020-05-14,ADBE,0,2321,1,0.075354,0.129295,0.301612,Upgrades\r\nDowngrades\r\nInitiations\r\n© 202...,neutral,0.932666,0,0,1,0
3,"DZ Bank Downgrades Adobe to Hold, Announces $3...",https://www.benzinga.com/news/20/05/16029565/d...,Vick Meyer,2020-05-14,ADBE,0,1968,1,0.075354,0.129295,0.301612,Never miss a trade again with the fastest news...,neutral,0.932860,0,0,1,0
4,"BMO Capital Maintains Outperform on Adobe, Rai...",https://www.benzinga.com/news/20/04/15921434/b...,Vick Meyer,2020-04-30,ADBE,0,1968,1,0.037156,0.101911,0.303897,Never miss a trade again with the fastest news...,neutral,0.932860,0,0,1,0
